# Trainer comparison notebook

This notebook validates and compares four real trainer paths:

- `PrioritySearch` as the Trace baseline
- `TextGradTrainer`
- `OpenEvolveTrainer`
- `DSPyTrainer`

It checks out the `textgrad_openevolve` branch, installs the real optional packages when needed, runs focused structural checks, and then runs a tiny real train/test comparison with OpenRouter or OpenAI.
The notebook assumes the `textgrad_openevolve` branch contains the trainer integration under test.

## What this notebook verifies

- required trainer packages import from real installations
- Trace-Bench discovers the trainer classes
- focused tests and compile checks pass
- every comparison row uses three train examples and three held-out examples
- result tables show trainer status, optimizer identity, before/after scores, and per-example outputs

## High-level interpretation guide

Use this notebook in three layers:

1. **Code-level correctness**
   - Do the new trainers exist?
   - Are they discovered?
   - Do focused tests pass?

2. **Behavior-level smoke checks**
   - Do the trainer paths run against real installed packages?
   - Do they produce comparable before/after rows?

3. **Practical comparison**
   - Which trainers improve on the tiny task?
   - Which trainers complete but do not improve in this small budget?

In [1]:
import os
import sys
import subprocess
from collections.abc import Sequence
from pathlib import Path
from subprocess import CompletedProcess

WORKDIR = Path("/content") if Path("/content").exists() else Path.cwd()
CURRENT_REPO = Path.cwd()
TRACE_BENCH_REMOTE_URL = "https://github.com/doxav/Trace-Bench.git"
TRACE_BENCH_BRANCH = "textgrad_openevolve"
TRACE_BENCH_REPO = CURRENT_REPO if (CURRENT_REPO / "trace_bench").is_dir() else WORKDIR / "Trace-Bench"
NEWTRACE_REMOTE_URL = "https://github.com/doxav/NewTrace.git"
NEWTRACE_BRANCH = "experimental"
NEWTRACE_REPO = WORKDIR / "NewTrace"
OPENEVOLVE_REMOTE_URL = "https://github.com/algorithmicsuperintelligence/openevolve.git"
OPENEVOLVE_BRANCH = "main"
OPENEVOLVE_REPO = WORKDIR / "openevolve"

for repo_path in (NEWTRACE_REPO, TRACE_BENCH_REPO):
    repo_path_str = str(repo_path)
    if repo_path_str not in sys.path:
        sys.path.insert(0, repo_path_str)

def run(cmd: Sequence[str | os.PathLike[str]], cwd: Path | str | None = None, check: bool = True) -> CompletedProcess[bytes]:
    """Run a subprocess command and echo its argv without shell interpolation."""
    print("$", " ".join(map(str, cmd)))
    return subprocess.run([str(part) for part in cmd], cwd=cwd, check=check)

def checkout_branch(repo_path: Path, remote_url: str, branch: str) -> None:
    """Fetch, checkout, and fast-forward a branch in an existing clone."""
    run(["git", "fetch", remote_url, branch], cwd=repo_path)
    checkout = run(["git", "checkout", branch], cwd=repo_path, check=False)
    if checkout.returncode != 0:
        run(["git", "checkout", "-b", branch, "FETCH_HEAD"], cwd=repo_path)
    run(["git", "pull", "--ff-only", remote_url, branch], cwd=repo_path)

print("WORKDIR =", WORKDIR)
print("TRACE_BENCH_REMOTE_URL =", TRACE_BENCH_REMOTE_URL)
print("TRACE_BENCH_BRANCH =", TRACE_BENCH_BRANCH)
print("TRACE_BENCH_REPO =", TRACE_BENCH_REPO)
print("NEWTRACE_REMOTE_URL =", NEWTRACE_REMOTE_URL)
print("NEWTRACE_BRANCH =", NEWTRACE_BRANCH)
print("NEWTRACE_REPO =", NEWTRACE_REPO)
print("OPENEVOLVE_REMOTE_URL =", OPENEVOLVE_REMOTE_URL)
print("OPENEVOLVE_BRANCH =", OPENEVOLVE_BRANCH)
print("OPENEVOLVE_REPO =", OPENEVOLVE_REPO)

WORKDIR = /home/xav/code/Trace-Bench
TRACE_BENCH_REMOTE_URL = https://github.com/doxav/Trace-Bench.git
TRACE_BENCH_BRANCH = textgrad_openevolve
TRACE_BENCH_REPO = /home/xav/code/Trace-Bench
NEWTRACE_REMOTE_URL = https://github.com/doxav/NewTrace.git
NEWTRACE_BRANCH = experimental
NEWTRACE_REPO = /home/xav/code/Trace-Bench/NewTrace
OPENEVOLVE_REMOTE_URL = https://github.com/algorithmicsuperintelligence/openevolve.git
OPENEVOLVE_BRANCH = main
OPENEVOLVE_REPO = /home/xav/code/Trace-Bench/openevolve


## 1. Clone and checkout the repositories

This clones:
- `Trace-Bench` on `textgrad_openevolve`
- `doxav/NewTrace` on `experimental`
- `OpenEvolve` only if the real package is missing

Skip this if you already have local checkouts and want to point the notebook at them manually.

In [2]:
if not TRACE_BENCH_REPO.exists():
    run([
        "git", "clone",
        "--branch", TRACE_BENCH_BRANCH,
        "--single-branch",
        TRACE_BENCH_REMOTE_URL,
        str(TRACE_BENCH_REPO),
    ])
else:
    print(f"Trace-Bench already exists; checking out {TRACE_BENCH_BRANCH}.")
    checkout_branch(TRACE_BENCH_REPO, TRACE_BENCH_REMOTE_URL, TRACE_BENCH_BRANCH)

if not NEWTRACE_REPO.exists():
    run([
        "git", "clone",
        "--branch", NEWTRACE_BRANCH,
        "--single-branch",
        NEWTRACE_REMOTE_URL,
        str(NEWTRACE_REPO),
    ])
else:
    print(f"NewTrace already exists; checking out {NEWTRACE_BRANCH}.")
    checkout_branch(NEWTRACE_REPO, NEWTRACE_REMOTE_URL, NEWTRACE_BRANCH)

Trace-Bench already exists; checking out textgrad_openevolve.
$ git fetch https://github.com/doxav/Trace-Bench.git textgrad_openevolve


From https://github.com/doxav/Trace-Bench
 * branch            textgrad_openevolve -> FETCH_HEAD
Already on 'textgrad_openevolve'


$ git checkout textgrad_openevolve
M	tests/test_openevolve_trainer.py
M	trace_bench/trainers/openevolve_trainer.py
Your branch is up to date with 'origin/textgrad_openevolve'.
$ git pull --ff-only https://github.com/doxav/Trace-Bench.git textgrad_openevolve


From https://github.com/doxav/Trace-Bench
 * branch            textgrad_openevolve -> FETCH_HEAD


Already up to date.
NewTrace already exists; checking out experimental.
$ git fetch https://github.com/doxav/NewTrace.git experimental


From https://github.com/doxav/NewTrace
 * branch            experimental -> FETCH_HEAD
Already on 'experimental'


$ git checkout experimental
Your branch is up to date with 'origin/experimental'.
$ git pull --ff-only https://github.com/doxav/NewTrace.git experimental


Already up to date.


From https://github.com/doxav/NewTrace
 * branch            experimental -> FETCH_HEAD


## 2. Install Python dependencies

This installs:
- `NewTrace` editable
- `Trace-Bench` editable
- light dependencies needed for the focused validation notebook

If `openevolve.run_evolution` is not importable, this clones OpenEvolve from GitHub and installs it editable.

In [3]:
run([sys.executable, "-m", "pip", "install", "-q", "-U", "pip", "setuptools", "wheel"])
run([sys.executable, "-m", "pip", "install", "-q",
     "graphviz", "pyyaml", "pytest", "litellm", "aiohttp", "nest_asyncio", "dspy-ai",
     "tensorboard", "tensorboardX", "scikit-learn", "datasets", "openai", "pandas"])

run([sys.executable, "-m", "pip", "install", "-q", "-e", str(NEWTRACE_REPO)])
run([sys.executable, "-m", "pip", "install", "-q", "-e", str(TRACE_BENCH_REPO)])

def has_real_openevolve() -> bool:
    """Return True only when the real OpenEvolve API is importable."""
    try:
        import openevolve
        return callable(getattr(openevolve, "run_evolution", None))
    except Exception:
        return False

if not has_real_openevolve():
    if not OPENEVOLVE_REPO.exists():
        run([
            "git", "clone",
            "--branch", OPENEVOLVE_BRANCH,
            "--single-branch",
            OPENEVOLVE_REMOTE_URL,
            str(OPENEVOLVE_REPO),
        ])
    else:
        print(f"OpenEvolve already exists; checking out {OPENEVOLVE_BRANCH}.")
        checkout_branch(OPENEVOLVE_REPO, OPENEVOLVE_REMOTE_URL, OPENEVOLVE_BRANCH)
    run([sys.executable, "-m", "pip", "install", "-q", "-e", str(OPENEVOLVE_REPO)])

if not has_real_openevolve():
    raise ImportError("OpenEvolve is required for this demo and could not be installed.")


$ /home/xav/miniconda3/bin/python -m pip install -q -U pip setuptools wheel


$ /home/xav/miniconda3/bin/python -m pip install -q graphviz pyyaml pytest litellm aiohttp nest_asyncio dspy-ai tensorboard tensorboardX scikit-learn datasets openai pandas


$ /home/xav/miniconda3/bin/python -m pip install -q -e /home/xav/code/Trace-Bench/NewTrace


$ /home/xav/miniconda3/bin/python -m pip install -q -e /home/xav/code/Trace-Bench


## 3. Provider setup for real online experiments

The real smoke comparison requires this provider setup. Structural tests can still run before a provider is configured.

Supported:
- `openrouter`
- `openai`

In [4]:
from getpass import getpass

def colab_secret(name: str) -> str:
    """Return a Colab Secret value when available, otherwise an empty string."""
    try:
        from google.colab import userdata
    except Exception:
        return ""
    try:
        return userdata.get(name) or ""
    except Exception:
        return ""

PROVIDER = "auto"  # @param ["auto", "openrouter", "openai", "none"]
MODEL = ""  # @param {type:"string"}

openrouter_key = os.environ.get("OPENROUTER_API_KEY") or colab_secret("OPENROUTER_API_KEY")
openai_key = os.environ.get("OPENAI_API_KEY") or colab_secret("OPENAI_API_KEY")
MODEL = MODEL or os.environ.get("TRACE_LITELLM_MODEL") or colab_secret("TRACE_LITELLM_MODEL")

if PROVIDER == "auto":
    active_provider = "openrouter" if openrouter_key else "openai" if openai_key else "none"
else:
    active_provider = PROVIDER

if active_provider == "openrouter":
    if not MODEL:
        MODEL = "openrouter/openai/gpt-4o-mini"
    if not openrouter_key:
        openrouter_key = getpass("OPENROUTER_API_KEY: ")
    if not openrouter_key:
        raise ValueError("OPENROUTER_API_KEY is required when PROVIDER is openrouter.")
    os.environ["OPENROUTER_API_KEY"] = openrouter_key
    os.environ["OPENAI_API_KEY"] = openrouter_key
    os.environ["OPENAI_BASE_URL"] = "https://openrouter.ai/api/v1"
    os.environ["OPENAI_API_BASE"] = "https://openrouter.ai/api/v1"
    os.environ["TRACE_LITELLM_MODEL"] = MODEL
elif active_provider == "openai":
    if not MODEL:
        MODEL = "gpt-4o-mini"
    if not openai_key:
        openai_key = getpass("OPENAI_API_KEY: ")
    if not openai_key:
        raise ValueError("OPENAI_API_KEY is required when PROVIDER is openai.")
    os.environ["OPENAI_API_KEY"] = openai_key
    os.environ["TRACE_LITELLM_MODEL"] = MODEL
elif active_provider == "none":
    print("Skipping online provider configuration.")
else:
    raise ValueError(f"Unsupported PROVIDER: {PROVIDER}")

print("PROVIDER =", active_provider)
print("TRACE_LITELLM_MODEL =", os.environ.get("TRACE_LITELLM_MODEL"))
print("OPENAI_BASE_URL =", os.environ.get("OPENAI_BASE_URL"))
print("OPENROUTER_API_KEY configured =", bool(os.environ.get("OPENROUTER_API_KEY")))

PROVIDER = openrouter
TRACE_LITELLM_MODEL = openrouter/openai/gpt-4o-mini
OPENAI_BASE_URL = https://openrouter.ai/api/v1
OPENROUTER_API_KEY configured = True


## 4. Sanity checks and imports

In [5]:
import importlib
import pandas as pd

def required_import(name: str) -> object:
    """Import a required module and raise a descriptive error when unavailable."""
    try:
        module = importlib.import_module(name)
        print("OK:", name)
        return module
    except Exception as exc:
        raise ImportError(f"Required module is unavailable: {name}") from exc

textgrad_module = required_import("opto.optimizers.textgrad")
openevolve_module = required_import("openevolve")
dspy_module = required_import("dspy")
required_import("trace_bench")
required_import("trace_bench.runner")
required_import("trace_bench.registry")
required_import("trace_bench.config")
required_import("trace_bench.trainers.textgrad_trainer")
required_import("trace_bench.trainers.openevolve_trainer")
required_import("trace_bench.trainers.dspy_trainer")

if not callable(getattr(textgrad_module, "TextGrad", None)):
    raise ImportError("opto.optimizers.textgrad.TextGrad is required for this demo.")
if not callable(getattr(openevolve_module, "run_evolution", None)):
    raise ImportError("openevolve.run_evolution is required for this demo.")
if not callable(getattr(dspy_module, "LM", None)):
    raise ImportError("dspy.LM is required for this demo.")

print("TextGrad module:", getattr(textgrad_module, "__file__", "unknown"))
print("OpenEvolve module:", getattr(openevolve_module, "__file__", "unknown"))
print("DSPy module:", getattr(dspy_module, "__file__", "unknown"))

OK: opto.optimizers.textgrad
OK: openevolve


OK: dspy
OK: trace_bench
OK: trace_bench.runner
OK: trace_bench.registry
OK: trace_bench.config
OK: trace_bench.trainers.textgrad_trainer
OK: trace_bench.trainers.openevolve_trainer
OK: trace_bench.trainers.dspy_trainer
TextGrad module: /home/xav/code/Trace-Bench/NewTrace/opto/optimizers/textgrad.py
OpenEvolve module: /home/xav/miniconda3/lib/python3.13/site-packages/openevolve/__init__.py
DSPy module: /home/xav/miniconda3/lib/python3.13/site-packages/dspy/__init__.py


## 5. Focused validation commands

These are the most relevant tests for the new trainers and their integration surface.

In [6]:
TARGETED_TESTS = [
    "tests/test_resolve_external_trainers.py",
    "tests/test_external_utils.py",
]

run([sys.executable, "-m", "pytest", *TARGETED_TESTS, "-q"], cwd=TRACE_BENCH_REPO)
run([sys.executable, "-m", "py_compile",
     "trace_bench/resolve.py",
     "trace_bench/cli.py",
     "trace_bench/runner.py",
     "trace_bench/trainers/_external_utils.py",
     "trace_bench/trainers/textgrad_trainer.py",
     "trace_bench/trainers/openevolve_trainer.py",
     "trace_bench/trainers/dspy_trainer.py"], cwd=TRACE_BENCH_REPO)

$ /home/xav/miniconda3/bin/python -m pytest tests/test_resolve_external_trainers.py tests/test_external_utils.py -q


...                                                                      [100%]
3 passed in 0.93s
$ /home/xav/miniconda3/bin/python -m py_compile trace_bench/resolve.py trace_bench/cli.py trace_bench/runner.py trace_bench/trainers/_external_utils.py trace_bench/trainers/textgrad_trainer.py trace_bench/trainers/openevolve_trainer.py trace_bench/trainers/dspy_trainer.py


CompletedProcess(args=['/home/xav/miniconda3/bin/python', '-m', 'py_compile', 'trace_bench/resolve.py', 'trace_bench/cli.py', 'trace_bench/runner.py', 'trace_bench/trainers/_external_utils.py', 'trace_bench/trainers/textgrad_trainer.py', 'trace_bench/trainers/openevolve_trainer.py', 'trace_bench/trainers/dspy_trainer.py'], returncode=0)

## 6. Trainer discovery and signatures

This is the fastest way to see whether the branch contains the trainer code and wires it into discovery.

In [7]:
from trace_bench.registry import discover_trainers
from trace_bench.runner import _resolve_algorithm

trainer_rows = []
for spec in discover_trainers():
    if spec.id in {"PrioritySearch", "TextGradTrainer", "OpenEvolveTrainer", "DSPyTrainer"}:
        resolved = _resolve_algorithm(spec.id)
        trainer_rows.append({
            "trainer_id": spec.id,
            "available": spec.available,
            "source": spec.source,
            "resolved_type": type(resolved).__name__ if not isinstance(resolved, type) else "class",
            "resolved_name": getattr(resolved, "__name__", str(resolved)),
            "uses_trace_optimizer": getattr(resolved, "USES_TRACE_OPTIMIZER", None) if isinstance(resolved, type) else None,
            "framework": getattr(resolved, "FRAMEWORK", None) if isinstance(resolved, type) else None,
        })

pd.DataFrame(trainer_rows).sort_values("trainer_id").reset_index(drop=True)

,trainer_id,available,source,resolved_type,resolved_name,uses_trace_optimizer,framework
0,DSPyTrainer,True,trace_bench.trainers.dspy_trainer,class,DSPyTrainer,False,dspy
1,OpenEvolveTrainer,True,trace_bench.trainers.openevolve_trainer,class,OpenEvolveTrainer,False,NaN
2,PrioritySearch,True,opto.features.priority_search.priority_search,str,PrioritySearch,None,NaN
3,TextGradTrainer,True,trace_bench.trainers.textgrad_trainer,class,TextGradTrainer,False,NaN


## 7. Shared helpers for train/test smoke evaluation

The Trace, TextGrad, and OpenEvolve rows reuse `trace_examples:opentrace_train_single_node`. The DSPy row uses a tiny real `dspy.Module` with the same scalar target. Every row learns from three examples and reports held-out performance on three more examples.

In [8]:
import re
from typing import Any

import dspy

from trace_bench.config import TrainerConfig
from trace_bench.registry import load_task_bundle
from trace_bench.runner import _train_bundle
from trace_bench.trainers._external_utils import apply_parameter_updates

TRACE_TASK_ID = "trace_examples:opentrace_train_single_node"
TASKS_ROOT = str(TRACE_BENCH_REPO / "LLM4AD" / "benchmark_tasks")
SMOKE_INITIAL_VALUE = 0.0
SMOKE_TARGET_VALUE = 3.0
SMOKE_TRAIN_DATASET = {
    "inputs": ["train-a", "train-b", "train-c"],
    "infos": [SMOKE_TARGET_VALUE, SMOKE_TARGET_VALUE, SMOKE_TARGET_VALUE],
}
SMOKE_TEST_DATASET = {
    "inputs": ["test-a", "test-b", "test-c"],
    "infos": [SMOKE_TARGET_VALUE, SMOKE_TARGET_VALUE, SMOKE_TARGET_VALUE],
}

class ScalarDSPySignature(dspy.Signature):
    """Always answer 0."""
    question: str = dspy.InputField()
    answer: str = dspy.OutputField(desc="numeric scalar answer only")

class ScalarDSPyAgent(dspy.Module):
    """Tiny DSPy module for the real DSPyTrainer smoke row."""
    def __init__(self) -> None:
        super().__init__()
        self.predict = dspy.Predict(ScalarDSPySignature)

    def forward(self, question: str) -> str:
        return self.predict(question=question).answer

    @classmethod
    def to_examples(cls, inputs: list[Any], infos: list[Any]) -> list[Any]:
        return [
            dspy.Example(question=str(task_input), answer=str(task_info), _task=task_input, _info=task_info).with_inputs("question")
            for task_input, task_info in zip(inputs, infos)
        ]

class ScalarDSPyGuide:
    """Score numeric DSPy answers against the scalar target."""
    def get_feedback(self, _query: Any, response: Any, reference: Any, **_kwargs: Any) -> tuple[float, str]:
        text = str(getattr(response, "data", response)).strip()
        match = re.search(r"-?\d+(?:\.\d+)?", text)
        prediction = float(match.group(0)) if match else float("nan")
        target = float(reference)
        score = -abs(prediction - target) if prediction == prediction else -10.0
        return score, f"target={target}; response={text}"

    def __call__(self, query: Any, response: Any, reference: Any, **kwargs: Any) -> tuple[float, str]:
        return self.get_feedback(query, response, reference, **kwargs)

def make_trace_bundle() -> dict[str, Any]:
    """Load the existing Trace-Bench scalar example bundle."""
    return load_task_bundle(TRACE_TASK_ID, TASKS_ROOT)

def _set_only_scalar_trainable(bundle: dict[str, Any]) -> None:
    """Keep the smoke focused on the existing scalar parameter."""
    param = bundle["param"]
    scalar = getattr(param, "value", None)
    if scalar is None:
        scalar = getattr(param, "guess", None)
    if scalar is None:
        raise AttributeError("Scalar smoke task requires param.value or param.guess.")
    for parameter in param.parameters():
        parameter.trainable = parameter is scalar
    apply_parameter_updates({scalar: SMOKE_INITIAL_VALUE})

def make_trace_smoke_bundle() -> dict[str, Any]:
    """Build a fresh train/test smoke bundle from the Trace scalar example."""
    bundle = make_trace_bundle()
    _set_only_scalar_trainable(bundle)
    bundle["train_dataset"] = SMOKE_TRAIN_DATASET
    bundle["test_dataset"] = SMOKE_TEST_DATASET
    bundle.pop("validate_dataset", None)
    bundle["optimizer_kwargs"]["objective"] = f"Set the trainable scalar to exactly {SMOKE_TARGET_VALUE}."
    bundle["metadata"]["task_label"] = "Trace scalar"
    return bundle

def make_dspy_lm() -> Any:
    """Build the real DSPy LM from the configured provider environment."""
    model = os.environ.get("TRACE_LITELLM_MODEL") or "gpt-4o-mini"
    if "/" not in model and ("gpt" in model.lower() or model.lower().startswith("o")):
        model = f"openai/{model}"
    lm_kwargs: dict[str, Any] = {"cache": False}
    api_base = os.environ.get("OPENAI_BASE_URL") or os.environ.get("OPENAI_API_BASE")
    if api_base:
        lm_kwargs["api_base"] = api_base
    return dspy.LM(model=model, **lm_kwargs)

def make_dspy_smoke_bundle() -> dict[str, Any]:
    """Build a fresh train/test smoke bundle for the DSPy trainer row."""
    dspy.configure(lm=make_dspy_lm())
    return {
        "param": ScalarDSPyAgent(),
        "guide": ScalarDSPyGuide(),
        "train_dataset": SMOKE_TRAIN_DATASET,
        "test_dataset": SMOKE_TEST_DATASET,
        "optimizer_kwargs": {"objective": f"Answer every scalar benchmark item with exactly {SMOKE_TARGET_VALUE}."},
        "metadata": {"task_label": "DSPy scalar", "framework": "dspy"},
    }

def short_text(value: Any, limit: int = 80) -> str:
    """Return a compact display value for comparison tables."""
    text = str(value)
    return text if len(text) <= limit else text[: limit - 3] + "..."

def snapshot_trainable_value(bundle: dict[str, Any]) -> Any:
    """Return the current scalar value or DSPy instruction."""
    scalar = getattr(bundle["param"], "value", None)
    if scalar is None:
        scalar = getattr(bundle["param"], "guess", None)
    if scalar is not None:
        return getattr(scalar, "data", None)
    predictor = getattr(bundle["param"], "predict", None)
    signature = getattr(predictor, "signature", None)
    return short_text(getattr(signature, "instructions", type(bundle["param"]).__name__))

def task_label(bundle: dict[str, Any]) -> str:
    """Return the display label for a smoke bundle."""
    return str(bundle.get("metadata", {}).get("task_label") or bundle.get("metadata", {}).get("benchmark") or "smoke")

def output_value(output: Any) -> Any:
    """Return a compact scalar/string output value."""
    return short_text(getattr(output, "data", output), limit=120)

def score_guide(guide: Any, task_input: Any, response: Any, task_info: Any) -> tuple[float, str]:
    """Score with Trace Guide or DSPy-style get_feedback guide."""
    if callable(guide):
        score, feedback = guide(task_input, response, task_info)
    else:
        score, feedback = guide.get_feedback(task_input, response, task_info)
    return float(score), str(feedback)

def run_train_bundle(
    trainer_id: str,
    params: dict[str, Any] | None = None,
    mode: str = "real",
    logger: str = "none",
    bundle_factory: Any = make_trace_smoke_bundle,
) -> dict[str, Any]:
    """Run one trainer on the 3-example train split and score the 3-example test split."""
    bundle = bundle_factory()
    params = params or {}
    before = {
        "value": snapshot_trainable_value(bundle),
        "train": score_dataset(bundle, SMOKE_TRAIN_DATASET),
        "test": score_dataset(bundle, SMOKE_TEST_DATASET),
    }
    result = _train_bundle(
        bundle=bundle,
        trainer_spec=TrainerConfig(id=trainer_id, params_variants=[params], logger=logger),
        params=params,
        mode=mode,
    )
    after = {
        "value": snapshot_trainable_value(bundle),
        "train": score_dataset(bundle, SMOKE_TRAIN_DATASET),
        "test": score_dataset(bundle, SMOKE_TEST_DATASET),
    }
    return {"trainer_id": trainer_id, "task": task_label(bundle), "mode": mode, "result": result, "before": before, "after": after}

def score_dataset(bundle: dict[str, Any], dataset: dict[str, list[Any]]) -> dict[str, Any]:
    """Evaluate a bundle on a dataset and retain per-example outputs."""
    inputs = dataset.get("inputs") or []
    infos = dataset.get("infos") or dataset.get("info") or []
    if len(inputs) != len(infos):
        raise ValueError("Dataset 'inputs' and 'infos' must have the same length.")
    if not inputs:
        raise ValueError("Dataset must contain at least one example.")

    rows = []
    scores = []
    for task_input, task_info in zip(inputs, infos):
        response = output_value(bundle["param"](task_input))
        score, feedback = score_guide(bundle["guide"], task_input, response, task_info)
        scores.append(score)
        rows.append({
            "input": task_input,
            "expected": task_info,
            "output": response,
            "score": score,
            "feedback": feedback,
        })
    return {"mean_score": sum(scores) / len(scores), "rows": rows}


## 8. Real train/test smoke runs

These runs use the real Trace-Bench trainer entry points and real installed trainer packages. They are intentionally tiny: small optimizer budgets, three training examples, and three held-out examples.

In [9]:
if active_provider == "none":
    raise RuntimeError("Real smoke comparison requires OPENROUTER_API_KEY or OPENAI_API_KEY.")

SMOKE_TRAINERS = [
    ("PrioritySearch", "Trace scalar", {"ps_steps": 1, "ps_batches": 1, "num_candidates": 2, "num_proposals": 2}, make_trace_smoke_bundle),
    ("TextGradTrainer", "Trace scalar", {"num_epochs": 1, "batch_size": 1, "ensure_improvement": True, "improvement_threshold": 1e-9, "max_tokens": 1024}, make_trace_smoke_bundle),
    ("OpenEvolveTrainer", "Trace scalar", {
        "iterations": 1,
        "ensure_improvement": True,
        "improvement_threshold": 1e-9,
        "verbose": False,
        "model": os.environ.get("TRACE_LITELLM_MODEL"),
        "api_base": os.environ.get("OPENAI_BASE_URL") or os.environ.get("OPENAI_API_BASE"),
        "api_key_env": "OPENAI_API_KEY",
        "max_tokens": 1024,
    }, make_trace_smoke_bundle),
    ("DSPyTrainer", "DSPy scalar", {
        "dspy_optimizer": "copro",
        "dspy_lm": make_dspy_lm(),
        "breadth": 2,
        "depth": 1,
        "num_threads": 1,
        "track_stats": False,
    }, make_dspy_smoke_bundle),
]

smoke_results = []
for trainer_id, task, params, bundle_factory in SMOKE_TRAINERS:
    try:
        smoke_results.append(run_train_bundle(trainer_id, params=params, mode="real", bundle_factory=bundle_factory))
    except Exception as exc:
        smoke_results.append({
            "trainer_id": trainer_id,
            "task": task,
            "mode": "real",
            "status": "error",
            "error": f"{type(exc).__name__}: {exc}",
        })

print(f"Completed {len(smoke_results)} real trainer smoke runs.")

PrioritySearch initialized with only long-term memory.
Epoch: 0. Iteration: 0


Sampling training minibatch: Sampling 2 agents on 1 inputs:   0%|          | 0/2 [00:00<?, ?it/s]

Sampling training minibatch: Sampling 2 agents on 1 inputs: 100%|██████████| 2/2 [00:00<00:00, 21024.08it/s]


/home/xav/code/Trace-Bench/NewTrace/opto/trainer/utils.py:76: RuntimeWarning: coroutine 'async_run.<locals>._run' was never awaited
  with concurrent.futures.ThreadPoolExecutor() as executor:


Evaluating agent:   0%|          | 0/3 [00:00<?, ?it/s]

...

Average Metric: -9.00 / 3 (-300.0%):  67%|██████▋   | 2/3 [00:04<00:01,  1.53s/it]

Average Metric: -9.00 / 3 (-300.0%): 100%|██████████| 3/3 [00:04<00:00,  1.40s/it]

Average Metric: -9.00 / 3 (-300.0%): 100%|██████████| 3/3 [00:04<00:00,  1.44s/it]

2026/05/28 19:31:16 INFO dspy.evaluate.evaluate: Average Metric: -9.0 / 3 (-300.0%)


Completed 4 real trainer smoke runs.


In [10]:
summary_rows = []
example_rows = []
for item in smoke_results:
    if "result" not in item:
        summary_rows.append({
            "trainer_id": item["trainer_id"],
            "task": item["task"],
            "mode": item["mode"],
            "status": item["status"],
            "resolved_optimizer": None,
            "before_value": None,
            "after_value": None,
            "train_examples": len(SMOKE_TRAIN_DATASET["inputs"]),
            "test_examples": len(SMOKE_TEST_DATASET["inputs"]),
            "before_train_score": None,
            "after_train_score": None,
            "train_delta": None,
            "before_test_score": None,
            "after_test_score": None,
            "test_delta": None,
            "error": item["error"],
        })
        continue
    result_status = item["result"].get("status")
    before_train = item["before"]["train"]["mean_score"]
    after_train = item["after"]["train"]["mean_score"]
    before_test = item["before"]["test"]["mean_score"]
    after_test = item["after"]["test"]["mean_score"]
    summary_rows.append({
        "trainer_id": item["trainer_id"],
        "task": item["task"],
        "mode": item["mode"],
        "status": result_status,
        "resolved_optimizer": item["result"].get("resolved_optimizer"),
        "before_value": item["before"]["value"],
        "after_value": item["after"]["value"],
        "train_examples": len(SMOKE_TRAIN_DATASET["inputs"]),
        "test_examples": len(SMOKE_TEST_DATASET["inputs"]),
        "before_train_score": before_train,
        "after_train_score": after_train,
        "train_delta": after_train - before_train,
        "before_test_score": before_test,
        "after_test_score": after_test,
        "test_delta": after_test - before_test,
        "error": item["result"].get("error"),
    })
    for split_name in ("train", "test"):
        for phase in ("before", "after"):
            for index, row in enumerate(item[phase][split_name]["rows"]):
                example_rows.append({
                    "trainer_id": item["trainer_id"],
                    "task": item["task"],
                    "split": split_name,
                    "phase": phase,
                    "example": index,
                    "input": row["input"],
                    "expected": row["expected"],
                    "output": row["output"],
                    "score": row["score"],
                })

trainer_comparison = pd.DataFrame(summary_rows)
example_comparison = pd.DataFrame(example_rows)

display(trainer_comparison)
if example_rows:
    display(example_comparison.sort_values(["task", "split", "example", "trainer_id", "phase"]).reset_index(drop=True))
else:
    print("No per-example outputs were produced because all real trainer runs errored.")

,trainer_id,task,mode,status,resolved_optimizer,before_value,after_value,train_examples,test_examples,before_train_score,after_train_score,train_delta,before_test_score,after_test_score,test_delta,error
0,PrioritySearch,Trace scalar,real,ok,OptoPrimeV2,0.0,3.0,3,3,-3.0,0.0,3.0,-3.0,0.0,3.0,None
1,TextGradTrainer,Trace scalar,real,ok,opto.optimizers.textgrad.TextGrad,0.0,3.0,3,3,-3.0,0.0,3.0,-3.0,0.0,3.0,None
2,OpenEvolveTrainer,Trace scalar,real,ok,openevolve.run_evolution,0.0,0.0,3,3,-3.0,-3.0,0.0,-3.0,-3.0,0.0,None
3,DSPyTrainer,DSPy scalar,real,ok,dspy.COPRO,Always answer 0.,"Provide a consistent response of ""0"" for any a...",3,3,-3.0,-3.0,0.0,-3.0,-3.0,0.0,None


,trainer_id,task,split,phase,example,input,expected,output,score
0,DSPyTrainer,DSPy scalar,test,after,0,test-a,3.0,0,-3.0
1,DSPyTrainer,DSPy scalar,test,before,0,test-a,3.0,0,-3.0
2,DSPyTrainer,DSPy scalar,test,after,1,test-b,3.0,0,-3.0
3,DSPyTrainer,DSPy scalar,test,before,1,test-b,3.0,0,-3.0
4,DSPyTrainer,DSPy scalar,test,after,2,test-c,3.0,0,-3.0
5,DSPyTrainer,DSPy scalar,test,before,2,test-c,3.0,0,-3.0
6,DSPyTrainer,DSPy scalar,train,after,0,train-a,3.0,0,-3.0
7,DSPyTrainer,DSPy scalar,train,before,0,train-a,3.0,0,-3.0
8,DSPyTrainer,DSPy scalar,train,after,1,train-b,3.0,0,-3.0
9,DSPyTrainer,DSPy scalar,train,before,1,train-b,3.0,0,-3.0


## 9. Practical reading guide

When you inspect the results, read them in this order:

1. **Focused tests**  
   If these fail, the branch is not ready to trust.

2. **Discovery table**  
   If `TextGradTrainer`, `OpenEvolveTrainer`, or `DSPyTrainer` are missing, the branch or optional packages are not properly present or installed.

3. **Real train/test smoke tables**  
   This confirms each trainer uses the real installed package path on three train examples and three held-out examples.

4. **Error rows**  
   An error row means the real trainer path failed and should be inspected before trusting comparison scores.

## 10. What counts as success

### Strong success
- focused tests pass
- discovery shows the comparison trainers
- real `opto.optimizers.textgrad.TextGrad`, `openevolve.run_evolution`, and `dspy.LM` import successfully
- real smoke rows for Trace, TextGrad, OpenEvolve, and DSPy complete without errors

### Partial success
- focused tests pass
- structural checks pass
- one trainer reports an error row while the others complete, making the failure comparable

### Failure
- trainers are not discovered
- focused tests fail
- DSPy is not backed by a real `dspy.LM`
- OpenEvolve path requires `exec` or unsafe parsing